# sample3 - Hugging Face Transformers 基礎

**Hugging Face Transformers** は事前学習済みモデルを簡単に使えるライブラリです。  
BERT・GPT・T5 など数万のモデルが公開されています。

| ステップ | 内容 |
|----------|------|
| 1 | pipeline で手軽にタスクを実行 |
| 2 | テキスト分類 |
| 3 | テキスト生成 |
| 4 | 要約 |
| 5 | 埋め込みベクトルの取得 |

In [1]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print("使用デバイス:", 'GPU' if device == 0 else 'CPU')

使用デバイス: GPU


## 1. pipeline で手軽にタスクを実行

`pipeline` は前処理・モデル推論・後処理をまとめて行う高レベル API です。  
初回実行時にモデルが自動ダウンロードされます。

## 2. テキスト分類（感情分析）

In [2]:
# 感情分析パイプライン（初回はモデルをダウンロード）
classifier = pipeline('sentiment-analysis', device=device)

texts = [
    "I love PyTorch! It's amazing.",
    "This is terrible and frustrating.",
    "The weather is okay today."
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"テキスト: {text}")
    print(f"結果    : {result['label']} (スコア: {result['score']:.4f})")
    print()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

テキスト: I love PyTorch! It's amazing.
結果    : POSITIVE (スコア: 0.9999)

テキスト: This is terrible and frustrating.
結果    : NEGATIVE (スコア: 0.9996)

テキスト: The weather is okay today.
結果    : POSITIVE (スコア: 0.9998)



## 3. テキスト生成

In [3]:
generator = pipeline('text-generation', model='gpt2', device=device)

result = generator(
    "Machine learning is",
    max_new_tokens=50,
    num_return_sequences=2,  # 2パターン生成
    do_sample=True,
    temperature=0.7
)

for i, r in enumerate(result):
    print(f"パターン {i+1}: {r['generated_text']}")
    print()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


パターン 1: Machine learning is a fundamental part of many software development strategies. The basic idea is to create a machine learning model that does not rely on the input from a human. This model is then used to create a simple process for determining which inputs are correct. You can see

パターン 2: Machine learning is one of the most promising opportunities for companies looking to develop and market their products and services.

But while there are many great things that can be done with this approach, it is really only a matter of time before the data comes in handy.



## 4. 要約

In [7]:
generator = pipeline('text-generation', model='gpt2', device=device)

prompt = "Summarize in one sentence: PyTorch is an open source deep learning framework developed by Meta AI, known for its flexibility and dynamic computation graph."

result = generator(
    prompt,
    max_new_tokens=50,
    do_sample=False
)
print("要約風出力:")
print(result[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


要約風出力:
Summarize in one sentence: PyTorch is an open source deep learning framework developed by Meta AI, known for its flexibility and dynamic computation graph.

PyTorch is a Python library for deep learning. It is a Python library for deep learning. It is a Python library for deep learning. It is a Python library for deep learning. It is a Python library for deep learning. It


## 5. 埋め込みベクトルの取得

テキストを数値ベクトルに変換します。RAG（sample5）で使う重要な技術です。

In [8]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import matplotlib.pyplot as plt

# sentence-transformers の軽量モデル
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] トークンの出力をベクトルとして使用
    return outputs.last_hidden_state[:, 0, :].numpy()[0]

sentences = [
    "PyTorch is a deep learning framework.",
    "TensorFlow is used for machine learning.",
    "I love eating sushi."
]

embeddings = [get_embedding(s) for s in sentences]
print(f"埋め込みベクトルの次元数: {len(embeddings[0])}")

# コサイン類似度で比較
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nコサイン類似度:")
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  '{sentences[i][:30]}...' vs '{sentences[j][:30]}...': {sim:.4f}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


埋め込みベクトルの次元数: 384

コサイン類似度:
  'PyTorch is a deep learning fra...' vs 'TensorFlow is used for machine...': 0.8320
  'PyTorch is a deep learning fra...' vs 'I love eating sushi....': 0.4474
  'TensorFlow is used for machine...' vs 'I love eating sushi....': 0.4886


# ダウンロードしたモデルを削除

In [10]:
import shutil
import os

hf_cache = os.path.expanduser('~/.cache/huggingface/hub')

models_to_delete = [
    'models--distilbert--distilbert-base-uncased-finetuned-sst-2-english',
    'models--facebook--bart-large-cnn',
    'models--gpt2',
    'models--sentence-transformers--all-MiniLM-L6-v2',
]

for model_dir in models_to_delete:
    path = os.path.join(hf_cache, model_dir)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"削除: {model_dir}")
    else:
        print(f"なし: {model_dir}")

削除: models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
削除: models--facebook--bart-large-cnn
なし: models--gpt2
なし: models--sentence-transformers--all-MiniLM-L6-v2
